# Piloto controlado de carga AXIAL Alkafri

Implementa una carga controlada, reanudable e idempotente de los 1.162 archivos AXIAL definidos en el manifiesto validado. La ejecucion por defecto es dry-run y no escribe en GCS.

## Configuracion segura

`UPLOAD_ENABLED=False`, `CONFIRM_UPLOAD=""`, `TARGET_FILE_COUNT=10`, `MAX_FILES_THIS_RUN=10`. No se reconstruyen rutas AXIAL; solo se usa el manifiesto.

## Validacion local previa

Antes de autenticar GCP se valida SHA, conteos, tamanios, rutas, componentes, ZIP excluidos y los 2.058 archivos locales con progreso cada 250.

## Estado previo y bucket

Se exige receipt compartido con 896 filas: 2 metadata, 447 spider_image, 447 spider_mask, 0 AXIAL, 447 pares SPIDER completos, 0 parciales, 0 fallas acumuladas y marcador `datasets/` vacio.

## Seleccion AXIAL y token

Selecciona determin?sticamente hasta 10 archivos AXIAL pendientes por orden original del manifiesto e imprime el token `UPLOAD_AXIAL_<N>_FILES_<BYTES>_BYTES_TO_<BUCKET>_<SHA12>`.

## Carga serial y verificacion

Si se arma manualmente, carga un archivo por vez con precondicion de generacion, checksum automatico, receipt atomico, state atomico y registro compatible de fallas.

## Resumen final

Imprime el JSON de auditoria con seleccion, carga, verificacion, AXIAL restante, SPIDER/metadata previos y estado de exito del lote.

In [25]:
from __future__ import annotations
import csv, hashlib, json, os, tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import pandas as pd

UPLOAD_ENABLED = False
CONFIRM_UPLOAD = ""

TARGET_FILE_COUNT = 10
MAX_FILES_THIS_RUN = 10
FAIL_FAST = True

TARGET_COMPONENTS = ("axial_image", "axial_mask")
PROJECT_ID = "pfi-asplanatti-fabrello-v1"
BUCKET_NAME = "pfi-rm-lumbar-artifacts-2026-ef"
BUCKET_URI_PREFIX = f"gs://{BUCKET_NAME}/"
GCS_PREFIX = "datasets/"
ALLOWED_ZERO_BYTE_PLACEHOLDERS = {GCS_PREFIX}
EXPECTED_BUCKET_LOCATION = "US-CENTRAL1"
EXPECTED_BUCKET_STORAGE_CLASS = "STANDARD"
EXPECTED_MANIFEST_SHA256 = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"
PLAN_DIR = PFI_ROOT / "results" / "GCS_dataset_migration_plan"
UPLOAD_STATE_DIR = PFI_ROOT / "results" / "GCS_dataset_upload"
MANIFEST_CSV = PLAN_DIR / "gcs_upload_manifest.csv"
SUMMARY_CSV = PLAN_DIR / "gcs_upload_summary.csv"
VALIDATION_JSON = PLAN_DIR / "gcs_upload_validation.json"
RECEIPT_CSV = UPLOAD_STATE_DIR / "gcs_upload_receipt.csv"
STATE_JSON = UPLOAD_STATE_DIR / "gcs_upload_state.json"
FAILURES_CSV = UPLOAD_STATE_DIR / "gcs_upload_failures.csv"
EXPECTED_MANIFEST_ROWS = 2058
EXPECTED_TOTAL_BYTES = 3922288649
EXPECTED_AXIAL_IMAGE_ROWS = 610
EXPECTED_AXIAL_MASK_ROWS = 552
EXPECTED_AXIAL_TOTAL_ROWS = 1162
EXPECTED_AXIAL_BYTES = 127819918
BASE_VERIFIED_RECEIPTS = 896
EXPECTED_METADATA_ROWS = 2
EXPECTED_SPIDER_IMAGE_RECEIPTS = 447
EXPECTED_SPIDER_MASK_RECEIPTS = 447
EXPECTED_SPIDER_PAIRS = 447
EXPECTED_MANIFEST_COLUMNS = ["component","source_path","source_root","relative_path","destination_uri","suffix","size_bytes","modified_at","sha256","referenced_rows","exists","is_symlink","status"]
VALID_COMPONENTS = {"spider_image","spider_mask","axial_image","axial_mask","metadata_e5","metadata_e9"}
RECEIPT_COLUMNS = ["manifest_sha256","component","source_path","relative_path","destination_uri","object_name","size_bytes","crc32c","md5_hash","generation","updated","uploaded_at_utc","upload_status"]
FAILURE_COLUMNS = ["manifest_sha256","component","relative_path","destination_uri","error_type","error_message","failed_at_utc"]

def ensure_drive_available():
    if not MY_DRIVE.exists():
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError("Colab o /content/drive/MyDrive requerido") from exc
        drive.mount(str(DRIVE_ROOT))
    if not PFI_ROOT.exists():
        raise FileNotFoundError(f"No existe PFI_ROOT: {PFI_ROOT}")

def sha256_stream(path: Path) -> str:
    with path.open("rb") as fh:
        return hashlib.file_digest(fh, "sha256").hexdigest()

def parse_bool_series(series: pd.Series, name: str) -> pd.Series:
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.strip().str.lower()
    bad = sorted(set(normalized) - {"true", "false"})
    if bad:
        raise ValueError(f"Columna {name} invalida: {bad[:10]}")
    return normalized == "true"

def path_is_inside(path_text: str, root: Path) -> bool:
    path = Path(path_text)
    return path.is_absolute() and (path == root or root in path.parents)

def destination_object_name(uri: str) -> str:
    if not isinstance(uri, str) or not uri.startswith(BUCKET_URI_PREFIX):
        raise ValueError(f"Destino fuera de bucket: {uri}")
    object_name = uri[len(BUCKET_URI_PREFIX):]
    if not object_name.startswith(GCS_PREFIX):
        raise ValueError(f"Destino fuera de prefijo {GCS_PREFIX}: {uri}")
    return object_name

def validate_manifest() -> tuple[pd.DataFrame, str]:
    for required in [MANIFEST_CSV, SUMMARY_CSV, VALIDATION_JSON]:
        if not required.exists() or not required.is_file():
            raise FileNotFoundError(required)
    manifest_sha256 = sha256_stream(MANIFEST_CSV)
    if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
        raise ValueError(f"Manifest SHA inesperado: {manifest_sha256}")
    m = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    if list(m.columns) != EXPECTED_MANIFEST_COLUMNS or len(m) != EXPECTED_MANIFEST_ROWS:
        raise ValueError("Manifest con columnas o filas inesperadas")
    m["manifest_ordinal"] = range(len(m))
    m["size_bytes"] = pd.to_numeric(m["size_bytes"], errors="raise").astype("int64")
    m["exists"] = parse_bool_series(m["exists"], "exists")
    m["is_symlink"] = parse_bool_series(m["is_symlink"], "is_symlink")
    checks = [int(m["size_bytes"].sum()) == EXPECTED_TOTAL_BYTES, bool((m["status"].astype(str) == "OK").all()), bool(m["exists"].all()), not bool(m["is_symlink"].any()), not bool((m["size_bytes"] <= 0).any()), not m["destination_uri"].duplicated().any(), not m.duplicated(["component","source_path"]).any(), not bool(m["suffix"].astype(str).str.lower().eq(".zip").any()), bool(m["component"].isin(VALID_COMPONENTS).all()), bool(m["source_path"].map(lambda v: path_is_inside(str(v), PFI_ROOT)).all()), bool(m["destination_uri"].astype(str).str.startswith(BUCKET_URI_PREFIX).all())]
    if bool(m["relative_path"].astype(str).str.strip().eq("").any()) or bool(m["relative_path"].astype(str).str.split("/").map(lambda parts: ".." in parts).any()):
        checks.append(False)
    if not all(checks):
        raise ValueError("Manifest invalido para piloto AXIAL")
    axial = m[m["component"].isin(TARGET_COMPONENTS)]
    counts = axial["component"].value_counts().to_dict()
    if int(counts.get("axial_image",0)) != EXPECTED_AXIAL_IMAGE_ROWS or int(counts.get("axial_mask",0)) != EXPECTED_AXIAL_MASK_ROWS or len(axial) != EXPECTED_AXIAL_TOTAL_ROWS or int(axial["size_bytes"].sum()) != EXPECTED_AXIAL_BYTES:
        raise ValueError("Conteos o bytes AXIAL inesperados")
    return m, manifest_sha256

def revalidate_sources_now(m: pd.DataFrame):
    total = len(m)
    for index, row in enumerate(m.itertuples(index=False), start=1):
        p = Path(row.source_path)
        if not p.exists() or p.is_symlink() or not p.is_file() or p.stat().st_size != int(row.size_bytes):
            raise ValueError(f"Fuente invalida: component={row.component} relative_path={row.relative_path}")
        if index % 250 == 0 or index == total:
            print(f"Fuentes revalidadas: {index}/{total}")

def load_receipt() -> pd.DataFrame:
    if not RECEIPT_CSV.exists() or not RECEIPT_CSV.is_file():
        raise FileNotFoundError(f"Falta receipt compartido: {RECEIPT_CSV}")
    r = pd.read_csv(RECEIPT_CSV, keep_default_na=False)
    if list(r.columns) != RECEIPT_COLUMNS:
        raise ValueError("Receipt incompatible")
    if r["destination_uri"].duplicated().any() or not bool((r["manifest_sha256"] == MANIFEST_SHA256).all()):
        raise ValueError("Receipt con destinos duplicados o SHA invalido")
    if not bool(r["component"].isin(VALID_COMPONENTS).all()):
        raise ValueError("Receipt contiene componentes no validos")
    if bool(r[["generation", "crc32c"]].astype(str).apply(lambda col: col.str.strip().eq("")).any().any()):
        raise ValueError("Receipt contiene generation/crc32c vacios")
    if not bool((r["upload_status"].astype(str) == "UPLOADED_VERIFIED").all()):
        raise ValueError("Receipt contiene upload_status no verificado")
    counts = r["component"].value_counts().to_dict()
    axial_images = int(counts.get("axial_image", 0))
    axial_masks = int(counts.get("axial_mask", 0))
    if int(counts.get("metadata_e5",0)) != 1 or int(counts.get("metadata_e9",0)) != 1:
        raise ValueError("Metadata previa incompleta")
    if int(counts.get("spider_image",0)) != EXPECTED_SPIDER_IMAGE_RECEIPTS or int(counts.get("spider_mask",0)) != EXPECTED_SPIDER_MASK_RECEIPTS:
        raise ValueError("SPIDER previo incompleto")
    if axial_images < 0 or axial_images > EXPECTED_AXIAL_IMAGE_ROWS or axial_masks < 0 or axial_masks > EXPECTED_AXIAL_MASK_ROWS:
        raise ValueError("Conteo AXIAL fuera de rango en receipt")
    expected_total = BASE_VERIFIED_RECEIPTS + axial_images + axial_masks
    if len(r) != expected_total:
        raise ValueError(f"Total de receipt inconsistente: {len(r)} != {expected_total}")
    manifest_lookup = manifest_df.copy()
    manifest_lookup["expected_object_name"] = manifest_lookup["destination_uri"].map(destination_object_name)
    expected = manifest_lookup.set_index("destination_uri")
    for row in r.itertuples(index=False):
        if row.destination_uri not in expected.index:
            raise ValueError(f"Receipt sin respaldo en manifest: {row.destination_uri}")
        exp = expected.loc[row.destination_uri]
        if str(row.component) != str(exp["component"]) or str(row.source_path) != str(exp["source_path"]) or str(row.relative_path) != str(exp["relative_path"]) or str(row.object_name) != str(exp["expected_object_name"]) or int(row.size_bytes) != int(exp["size_bytes"]):
            raise ValueError(f"Receipt no coincide con manifest: {row.destination_uri}")
    return r

def load_failures() -> pd.DataFrame:
    if not FAILURES_CSV.exists():
        return pd.DataFrame(columns=FAILURE_COLUMNS)
    f = pd.read_csv(FAILURES_CSV, keep_default_na=False)
    if list(f.columns) != FAILURE_COLUMNS:
        raise ValueError("Failures incompatible")
    return f

def authenticate_gcp():
    try:
        from google.colab import auth
    except ImportError as exc:
        raise RuntimeError("Autenticacion interactiva de Colab requerida") from exc
    try:
        import google.auth
        from google.cloud import storage
    except ImportError as exc:
        raise RuntimeError("Falta google-cloud-storage/google-auth; instalar manualmente") from exc
    auth.authenticate_user()
    credentials, auth_project = google.auth.default()
    print(json.dumps({"client_project": PROJECT_ID, "auth_project_detected": auth_project}, indent=2))
    return storage.Client(project=PROJECT_ID, credentials=credentials)

def planned_object_names(m: pd.DataFrame) -> set[str]:
    return set(m["destination_uri"].map(destination_object_name))

def inspect_bucket(client):
    bucket = client.bucket(BUCKET_NAME)
    bucket.reload(client=client)
    if bucket.name != BUCKET_NAME or str(bucket.location or "").upper() != EXPECTED_BUCKET_LOCATION or str(bucket.storage_class or "").upper() != EXPECTED_BUCKET_STORAGE_CLASS:
        raise ValueError("Metadata de bucket inesperada")
    planned = planned_object_names(manifest_df)
    receipt_objects = set(receipt_df["object_name"].astype(str))
    existing, placeholders, external = {}, [], []
    for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        item = {"name": blob.name, "size": int(blob.size or 0), "crc32c": blob.crc32c, "md5_hash": blob.md5_hash, "generation": str(blob.generation) if blob.generation is not None else None, "updated": blob.updated.isoformat() if blob.updated is not None else None}
        existing[blob.name] = item
        if blob.name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and item["size"] == 0:
            placeholders.append(item)
        elif blob.name not in planned:
            external.append(item)
    if external:
        raise RuntimeError(f"Objetos no planificados bajo {GCS_PREFIX}: {external[:10]}")
    if len(placeholders) != 1:
        raise RuntimeError("Placeholder datasets/ faltante o duplicado")
    unexpected_existing = sorted(set(existing) - receipt_objects - ALLOWED_ZERO_BYTE_PLACEHOLDERS)
    missing_receipt_objects = sorted(receipt_objects - set(existing))
    if unexpected_existing:
        raise RuntimeError(f"Objetos planificados existentes sin receipt confiable: {unexpected_existing[:10]}")
    if missing_receipt_objects:
        raise RuntimeError(f"Faltan objetos registrados en receipt: {missing_receipt_objects[:10]}")
    expected_object_count = len(receipt_df) + 1
    if len(existing) != expected_object_count:
        raise RuntimeError(f"Cantidad de objetos inconsistente: {len(existing)} != {expected_object_count}")
    for row in receipt_df.itertuples(index=False):
        remote = existing.get(str(row.object_name))
        if remote is None or int(remote["size"]) != int(row.size_bytes) or str(remote["generation"]) != str(row.generation) or str(remote["crc32c"]) != str(row.crc32c):
            raise RuntimeError(f"Objeto remoto no coincide con receipt: {row.destination_uri}")
    return bucket, existing, placeholders, external

def atomic_write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as fh:
            fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, path)
    finally:
        t = Path(tmp)
        if t.exists():
            t.unlink()

def save_receipt(receipt: pd.DataFrame):
    atomic_write_text(RECEIPT_CSV, receipt[RECEIPT_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))

def save_failures(failures: pd.DataFrame):
    atomic_write_text(FAILURES_CSV, failures[FAILURE_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))

def save_state(payload: dict[str, Any]):
    atomic_write_text(STATE_JSON, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def receipt_lookup(r: pd.DataFrame) -> dict[str, dict[str, Any]]:
    return {str(row.destination_uri): row._asdict() for row in r.itertuples(index=False)}

def remote_matches_receipt(row: pd.Series, remote: dict[str, Any] | None, rec: dict[str, Any] | None) -> bool:
    return bool(remote is not None and rec is not None and str(rec["manifest_sha256"]) == MANIFEST_SHA256 and str(rec["destination_uri"]) == str(row["destination_uri"]) and int(rec["size_bytes"]) == int(remote["size"]) and str(rec["generation"]) == str(remote["generation"]) and str(rec["crc32c"]) == str(remote["crc32c"]))

def classify_plan_status(m: pd.DataFrame, existing: dict[str, dict[str, Any]], receipt: pd.DataFrame) -> pd.DataFrame:
    receipts = receipt_lookup(receipt); rows = []
    for row in m.itertuples(index=False):
        rec = row._asdict(); object_name = destination_object_name(str(rec["destination_uri"])); remote = existing.get(object_name); receipt_row = receipts.get(str(rec["destination_uri"]))
        status = "TO_UPLOAD" if remote is None else ("ALREADY_UPLOADED_VERIFIED" if remote_matches_receipt(pd.Series(rec), remote, receipt_row) else "EXISTING_UNVERIFIED")
        rows.append({"manifest_ordinal": int(rec["manifest_ordinal"]), "component": rec["component"], "source_path": rec["source_path"], "relative_path": rec["relative_path"], "destination_uri": rec["destination_uri"], "object_name": object_name, "size_bytes": int(rec["size_bytes"]), "plan_status": status})
    return pd.DataFrame(rows)

def normalized_spider_pair_key(path_like: str) -> str:
    stem = Path(path_like).stem.lower()
    for suffix in ("_image", "_mask", "_label"):
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
    return stem

def validate_previous_state(classified: pd.DataFrame):
    verified = classified[classified["plan_status"] == "ALREADY_UPLOADED_VERIFIED"]
    vc = verified["component"].value_counts().to_dict()
    if int(vc.get("metadata_e5",0)) != 1 or int(vc.get("metadata_e9",0)) != 1:
        raise RuntimeError("Metadata verificada inesperada")
    if int(vc.get("spider_image",0)) != EXPECTED_SPIDER_IMAGE_RECEIPTS or int(vc.get("spider_mask",0)) != EXPECTED_SPIDER_MASK_RECEIPTS:
        raise RuntimeError("SPIDER verificado inesperado")
    spider = verified[verified["component"].isin(["spider_image","spider_mask"])].copy()
    spider["pair_key"] = spider["relative_path"].map(normalized_spider_pair_key)
    per_pair = spider.groupby(["pair_key","component"]).size().unstack(fill_value=0)
    full_pairs = int(((per_pair.get("spider_image",0) == 1) & (per_pair.get("spider_mask",0) == 1)).sum())
    partial_pairs = int(((per_pair.get("spider_image",0) + per_pair.get("spider_mask",0)) == 1).sum())
    if full_pairs != EXPECTED_SPIDER_PAIRS or partial_pairs != 0:
        raise RuntimeError("Pares SPIDER previos invalidos")
    verified_axial_images = int(vc.get("axial_image", 0))
    verified_axial_masks = int(vc.get("axial_mask", 0))
    verified_axial_files = verified_axial_images + verified_axial_masks
    return full_pairs, partial_pairs, verified_axial_files, verified_axial_images, verified_axial_masks

ensure_drive_available()
manifest_df, MANIFEST_SHA256 = validate_manifest()
revalidate_sources_now(manifest_df)
receipt_df = load_receipt()
failures_df = load_failures()
storage_client = authenticate_gcp()
bucket, existing_objects, allowed_placeholders, unplanned_existing_objects = inspect_bucket(storage_client)
classified_manifest_df = classify_plan_status(manifest_df, existing_objects, receipt_df)
existing_unverified_df = classified_manifest_df[classified_manifest_df["plan_status"] == "EXISTING_UNVERIFIED"]
if not existing_unverified_df.empty:
    raise RuntimeError(f"Objetos existentes sin receipt confiable: {existing_unverified_df.head(10).to_dict('records')}")
fully_verified_spider_pairs, partial_verified_spider_pairs, verified_axial_files_start, verified_axial_images_start, verified_axial_masks_start = validate_previous_state(classified_manifest_df)
axial_status_df = classified_manifest_df[classified_manifest_df["component"].isin(TARGET_COMPONENTS)].sort_values("manifest_ordinal", kind="stable").copy()
verified_axial_df = axial_status_df[axial_status_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED"]
pending_axial_df = axial_status_df[axial_status_df["plan_status"] == "TO_UPLOAD"]
selected_df = pending_axial_df.head(min(TARGET_FILE_COUNT, len(pending_axial_df))).copy()
selected_files = int(len(selected_df)); selected_bytes = int(selected_df["size_bytes"].sum()) if selected_files else 0
AXIAL_DATASET_COMPLETE = bool(verified_axial_files_start == EXPECTED_AXIAL_TOTAL_ROWS and selected_files == 0)
if selected_files > MAX_FILES_THIS_RUN or selected_df["destination_uri"].duplicated().any() or selected_df.duplicated(["component","source_path"]).any() or not bool((selected_df["plan_status"] == "TO_UPLOAD").all()) or not bool(selected_df["component"].isin(TARGET_COMPONENTS).all()):
    raise RuntimeError("Seleccion AXIAL invalida")
if selected_files == 0 and not AXIAL_DATASET_COMPLETE:
    raise RuntimeError("No hay seleccion AXIAL pero el dataset no esta completo")
if selected_files:
    print(selected_df[["manifest_ordinal","component","relative_path","destination_uri","size_bytes","plan_status"]].to_string(index=False))
else:
    print("AXIAL completo: no quedan archivos pendientes para cargar.")
CONFIRMATION_TOKEN = f"UPLOAD_AXIAL_{selected_files}_FILES_{selected_bytes}_BYTES_TO_{BUCKET_NAME}_{MANIFEST_SHA256[:12]}"
UPLOAD_ARMED = bool(UPLOAD_ENABLED is True and CONFIRM_UPLOAD == CONFIRMATION_TOKEN and selected_files > 0 and not AXIAL_DATASET_COMPLETE)
print(json.dumps({"required_confirmation_token": CONFIRMATION_TOKEN, "UPLOAD_ENABLED": UPLOAD_ENABLED, "UPLOAD_ARMED": UPLOAD_ARMED}, indent=2))
failures_this_run: list[dict[str, Any]] = []

def append_failure(failures: pd.DataFrame, row: pd.Series, error_type: str, error_message: str) -> pd.DataFrame:
    item = {"manifest_sha256": MANIFEST_SHA256, "component": row["component"], "relative_path": row["relative_path"], "destination_uri": row["destination_uri"], "error_type": error_type, "error_message": str(error_message)[:500], "failed_at_utc": datetime.now(timezone.utc).isoformat()}
    failures_this_run.append(item)
    failures = pd.concat([failures, pd.DataFrame([item])], ignore_index=True)
    save_failures(failures)
    return failures

def upload_one_axial_file(bucket: Any, row: pd.Series) -> dict[str, Any]:
    if row["component"] not in TARGET_COMPONENTS:
        raise ValueError(f"Componente no permitido: {row['component']}")
    source_path = Path(row["source_path"])
    if not source_path.exists() or source_path.is_symlink() or not source_path.is_file() or source_path.stat().st_size != int(row["size_bytes"]):
        raise ValueError(f"Fuente AXIAL invalida: {row['relative_path']}")
    object_name = destination_object_name(str(row["destination_uri"]))
    blob = bucket.blob(object_name)
    blob.metadata = {"pfi_manifest_sha256": MANIFEST_SHA256, "pfi_component": str(row["component"]), "pfi_relative_path": str(row["relative_path"]), "pfi_source_size": str(int(row["size_bytes"])), "pfi_upload_notebook": "42"}
    blob.upload_from_filename(
        str(source_path),
        if_generation_match=0,
        checksum="auto",
        timeout=900,
    )
    blob.reload()
    if int(blob.size or 0) != int(row["size_bytes"]) or blob.generation is None or blob.crc32c is None:
        raise ValueError(f"Verificacion remota fallo: {object_name}")
    return {"manifest_sha256": MANIFEST_SHA256, "component": row["component"], "source_path": row["source_path"], "relative_path": row["relative_path"], "destination_uri": row["destination_uri"], "object_name": object_name, "size_bytes": int(row["size_bytes"]), "crc32c": blob.crc32c, "md5_hash": blob.md5_hash, "generation": str(blob.generation), "updated": blob.updated.isoformat() if blob.updated is not None else "", "uploaded_at_utc": datetime.now(timezone.utc).isoformat(), "upload_status": "UPLOADED_VERIFIED"}

uploaded_receipts: list[dict[str, Any]] = []
if UPLOAD_ARMED:
    try:
        from google.api_core.exceptions import PreconditionFailed
        from google.cloud.storage.exceptions import DataCorruption
    except ImportError as exc:
        raise RuntimeError("Faltan excepciones de google-cloud-storage/google-api-core") from exc
    for row in selected_df.to_dict("records"):
        row_series = pd.Series(row)
        try:
            item = upload_one_axial_file(bucket, row_series)
            uploaded_receipts.append(item)
            receipt_df = pd.concat([receipt_df, pd.DataFrame([item])], ignore_index=True)
            save_receipt(receipt_df)
            save_state({"manifest_sha256": MANIFEST_SHA256, "notebook": "42", "last_uploaded_destination_uri": row["destination_uri"], "updated_at_utc": datetime.now(timezone.utc).isoformat()})
            print(f"Cargado y verificado: component={row['component']} relative_path={row['relative_path']}")
        except (PreconditionFailed, DataCorruption) as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            if FAIL_FAST: raise
        except Exception as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            if FAIL_FAST: raise
else:
    print("UPLOAD_ARMED=False; dry-run AXIAL sin escritura remota.")

def list_current_objects(client):
    return {blob.name: {"size": int(blob.size or 0), "crc32c": blob.crc32c, "generation": str(blob.generation) if blob.generation is not None else None} for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX)}

def verify_uploaded_batch(client, uploaded_items: list[dict[str, Any]]) -> int:
    if not uploaded_items: return 0
    current = list_current_objects(client); verified = 0
    for item in uploaded_items:
        remote = current.get(item["object_name"])
        if remote is None or int(remote["size"]) != int(item["size_bytes"]) or str(remote["generation"]) != str(item["generation"]) or str(remote["crc32c"]) != str(item["crc32c"]):
            raise RuntimeError(f"Verificacion final no coincide: {item['object_name']}")
        verified += 1
    return verified
verified_this_run = verify_uploaded_batch(storage_client, uploaded_receipts)
if uploaded_receipts:
    refreshed_receipt = pd.concat([receipt_df.iloc[0:0], receipt_df], ignore_index=True)
    refreshed_classified = classify_plan_status(manifest_df, list_current_objects(storage_client), refreshed_receipt)
else:
    refreshed_classified = classified_manifest_df
verified_axial_final = refreshed_classified[(refreshed_classified["component"].isin(TARGET_COMPONENTS)) & (refreshed_classified["plan_status"] == "ALREADY_UPLOADED_VERIFIED")]
verified_axial_images = int((verified_axial_final["component"] == "axial_image").sum())
verified_axial_masks = int((verified_axial_final["component"] == "axial_mask").sum())
verified_axial_files = int(len(verified_axial_final))
remaining_axial_files = int(EXPECTED_AXIAL_TOTAL_ROWS - verified_axial_files)
UPLOAD_BATCH_SUCCESS = bool(UPLOAD_ARMED and len(uploaded_receipts) == selected_files and verified_this_run == selected_files and len(failures_this_run) == 0)
final_summary = {"manifest_sha256": MANIFEST_SHA256, "target_components": list(TARGET_COMPONENTS), "target_file_count": TARGET_FILE_COUNT, "max_files_this_run": MAX_FILES_THIS_RUN, "selected_files": selected_files, "selected_bytes": selected_bytes, "uploaded_this_run": len(uploaded_receipts), "verified_this_run": verified_this_run, "already_uploaded_verified_files": int((classified_manifest_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED").sum()), "verified_axial_files": verified_axial_files, "verified_axial_images": verified_axial_images, "verified_axial_masks": verified_axial_masks, "remaining_axial_files": remaining_axial_files, "AXIAL_DATASET_COMPLETE": bool(remaining_axial_files == 0), "metadata_receipts_verified": True, "spider_receipts_verified": True, "fully_verified_spider_pairs": fully_verified_spider_pairs, "partial_verified_spider_pairs": partial_verified_spider_pairs, "failures_this_run": len(failures_this_run), "cumulative_failure_rows": int(len(failures_df)), "placeholder_count": len(allowed_placeholders), "UPLOAD_BATCH_SUCCESS": UPLOAD_BATCH_SUCCESS}
print(json.dumps(final_summary, indent=2))

Fuentes revalidadas: 250/2058
Fuentes revalidadas: 500/2058
Fuentes revalidadas: 750/2058
Fuentes revalidadas: 1000/2058
Fuentes revalidadas: 1250/2058
Fuentes revalidadas: 1500/2058
Fuentes revalidadas: 1750/2058
Fuentes revalidadas: 2000/2058
Fuentes revalidadas: 2058/2058
{
  "client_project": "pfi-asplanatti-fabrello-v1",
  "auth_project_detected": ""
}
 manifest_ordinal  component                                                                                             relative_path                                                                                                                                                       destination_uri  size_bytes plan_status
             2004 axial_mask extracted/_nested/ground_truth__Ground_Truth_Label/05_Final_Ground_Truth_Data/Label_Images/L1_0183_D5.png gs://pfi-rm-lumbar-artifacts-2026-ef/datasets/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/05_Final_Ground_Truth_Data/Label_Images/L1_0183_D5.png        1376   